In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykep

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from orbital_mechanics.solar_system import SolarSystem
from orbital_mechanics.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR

In [3]:
ss = SolarSystem()

# get id of PlanetX and rogue1
planetX = [pl for pl in ss.bodies if pl.name == 'PlanetX'][0]
rogue1 = [pl for pl in ss.bodies if pl.name == 'Rogue1'][0]

# spacecraft initial state at t0
import csv
with open('planetx_intercept.csv', 'r', newline='') as csvfile:
    last_state = list(csv.reader(csvfile))[-1]

last_state = np.array(last_state, dtype=np.float64)
t0 = last_state[2]
rv0 = last_state[3:9]
v_inf = last_state[9:12]

In [4]:
# dynamics function
def dydt(t,rv):
    r = rv[0:3]; v = rv[3:6]
    r_mag = np.linalg.norm(r)
    a = -MU/r_mag**3 * r
    dr = v; dv = a
    drv = np.concatenate([dr, dv])
    return drv

In [5]:
# distance from rogue1
def rel_rv_from_rogue1(t,rv):
    r = rv[0:3]; v = rv[3:6]

    # get rogue1's position and velocity
    pl_r, pl_v = rogue1.eph(t*pykep.SEC2DAY)

    rel_r = r - pl_r
    rel_v = v - pl_v

    return rel_r, rel_v

In [6]:
# event function (check periapsis distance to planetX)
def event(t,rv):
    rel_r, rel_v = rel_rv_from_rogue1(t,rv)
    res = np.dot(rel_r, rel_v)

    return res

event.terminal = True

In [7]:
res = solve_ivp(dydt, [t0, 200*YEAR], rv0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [8]:
vals = []
for i in range(len(res['t'])):
    rel_r, _ = rel_rv_from_rogue1(res['t'][i],res['y'][:,i].ravel())
    rel_r_mag = np.linalg.norm(rel_r)
    dotp = event(res['t'][i], res['y'][:,i].ravel())

    vals.append([res['t'][i], rel_r_mag, dotp])

vals = np.array(vals, dtype=float)
print(vals)

[[ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903204e+09  1.95619394e+10 -1.51410281e+11]
 [ 2.20903235e+09  1.95619370e+10 -1.51410262e+11]
 [ 2.20903554e+09  1.95619123e+10 -1.51410071e+11]
 [ 2.20906741e+09  1.95616657e+10 -1.51408166e+11]
 [ 2.20938608e+09  1.95591991e+10 -1.51389110e+11]
 [ 2.21257279e+09  1.95345338e+10 -1.51198490e+11]
 [ 2.24443991e+09  1.92878811e+10 -1.49286690e+11]
 [ 2.56311114e+09  1.68249608e+10 -1.29685961e+11]
 [ 2.84913665e+09  1.46307082e+10 -1.11571193e+11]
 [ 3.13516215e+09  1.24685938e+10 -9.32709535e+10]
 [ 3.36789826e+09  1.07473939e+10 -7.84477684e+10]
 [ 3.60063438e+09  9.07749917e+09 -6.38451486e+10]
 [ 3.79101889e+09  7.76691238e+09 -5.21540068e+10]
 [ 3.98140340e+09  6.53047245e+09 -4.07510296e+10]
 [ 4.15484244e+09  5.50204440e+09 -3.06429947e+10]
 [ 4.31131355e+09  4.69882085e+09 -2.17581975e+10]
 [ 4.45258311e+09  4.12832712e+

In [9]:
def polar2cart(mag, el, az):
    vx = mag * np.cos(el) * np.cos(az)
    vy = mag * np.cos(el) * np.sin(az)
    vz = mag * np.sin(el)

    dv = np.array([vx, vy, vz])

    return dv

In [10]:
# root func
def rootfunc(y0):
    # calculate delta V
    v0_mag = np.linalg.norm(v_inf)
    dv = polar2cart(v0_mag, y0[0], y0[1])

    # add delta V to planetX's velocity
    # get planetX's position and velocity
    _, pl_v = planetX.eph(t0*pykep.SEC2DAY)
    v0 = pl_v + dv

    yy0 = np.array([rv0[0], rv0[1], rv0[2], v0[0], v0[1], v0[2]])
    res = solve_ivp(dydt, [t0, 200*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)
    rel_r, _ = rel_rv_from_rogue1(res['t'][-1],res['y'][:,-1].ravel())
    print(res['t'][-1] / YEAR, rel_r)
    return rel_r

In [11]:
res = least_squares(rootfunc, [0.0, 0.0], bounds=([-np.pi/2, -np.pi], [np.pi/2, np.pi]))

134.0430193905569 [-1.08360925e+09  2.49267171e+07 -6.19315454e+09]
134.0430196409144 [-1.08360919e+09  2.49266730e+07 -6.19315441e+09]
134.04301972953652 [-1.08360917e+09  2.49268041e+07 -6.19315456e+09]
141.16529418409428 [-4.62197178e+08 -8.04553800e+08 -2.06011326e+09]
141.16529437307628 [-4.62197208e+08 -8.04553835e+08 -2.06011311e+09]
141.16529458521998 [-4.62197103e+08 -8.04553721e+08 -2.06011325e+09]
148.95079260516783 [-2.25952354e+08 -3.19153829e+08 -1.84122382e+08]
148.95079277179244 [-2.25952429e+08 -3.19153877e+08 -1.84122229e+08]
148.95079311839652 [-2.25952301e+08 -3.19153763e+08 -1.84122362e+08]
151.74326479314328 [-19343692.97737122 -22798504.07033098   2694806.37607354]
151.74326495322157 [-19343773.48391151 -22798556.91711116   2694963.13162166]
151.74326536709873 [-19343646.94314384 -22798441.58113229   2694829.01770878]
151.9126827209387 [-58618.41145706 -73338.99855685  -6126.1164937 ]
151.91268288092758 [-58698.88983536 -73392.18550515  -5969.02473629]
151.912683

In [12]:
res['message']

'`xtol` termination condition is satisfied.'

In [13]:
v1 = v_inf
v2 = polar2cart(np.linalg.norm(v_inf), res['x'][0], res['x'][1])

np.rad2deg(np.arccos(np.dot(v1, v2)/np.linalg.norm(v1)/np.linalg.norm(v2)))

np.float64(17.45239660642093)

In [14]:
def calc_min_max_th():
    rmu = float(rogue1.mu_self)
    rr = float(rogue1.radius)
    vinf_mag = float(np.linalg.norm(v_inf))
    sindel2 = (rmu/1.1/rr)/(vinf_mag + rmu/1.1/rr)
    delt = 2*np.arcsin(sindel2)
    print(np.rad2deg(delt))

calc_min_max_th()

164.639390303206


In [15]:
_, pl_v = planetX.eph(t0*pykep.SEC2DAY)
v2_i = pl_v + v2

In [16]:
yy0 = np.array([rv0[0], rv0[1], rv0[2], v2_i[0], v2_i[1], v2_i[2]])
res = solve_ivp(dydt, [t0, 200*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [17]:
res['t'] / YEAR

array([ 70.        ,  70.00000001,  70.0000001 ,  70.00000103,
        70.0000103 ,  70.00010301,  70.00103011,  70.0103011 ,
        70.09908425,  70.9869158 ,  79.86523132,  91.82831031,
       101.38967424, 109.970071  , 117.66315455, 124.56251705,
       130.75192209, 136.30739761, 141.29877146, 146.2901453 ,
       150.30226248, 151.91328711])

In [18]:
(res['y'][0:3]).T / AU

array([[-138.58320844,   10.67495418,  -28.02214553],
       [-138.58320844,   10.67495418,  -28.02214553],
       [-138.58320836,   10.67495417,  -28.0221455 ],
       [-138.58320761,   10.6749541 ,  -28.02214522],
       [-138.58320006,   10.6749534 ,  -28.02214239],
       [-138.58312462,   10.67494645,  -28.02211409],
       [-138.58237023,   10.67487689,  -28.02183107],
       [-138.57482622,   10.67418131,  -28.01900086],
       [-138.50257256,   10.66751946,  -27.9918957 ],
       [-137.7791584 ,   10.60083338,  -27.72066701],
       [-130.45349995,    9.92695435,  -24.99023072],
       [-120.29246289,    8.99690354,  -21.25600568],
       [-111.89449164,    8.23285044,  -18.22240305],
       [-104.11287504,    7.52914236,  -15.46010291],
       [ -96.90659726,    6.88161866,  -12.94939891],
       [ -90.2295333 ,    6.2856855 ,  -10.6691003 ],
       [ -84.03947187,    5.73713686,   -8.59979164],
       [ -78.29662399,    5.23202044,   -6.72331248],
       [ -72.96243115,    4.

In [19]:
rel_r, rel_v = rel_rv_from_rogue1(res['t'][-1], res['y'][:,-1].T)

In [20]:
rel_r, rel_v

(array([-0.0004406 , -0.00037217,  0.0005829 ]),
 array([ 5.54587253, -4.54156022,  1.30348063]))

In [21]:
planetx_id = planetX.gtoc13_id
rogue1_id = rogue1.gtoc13_id

# exit from the flyby:
sol_line_1_raw = [
    planetx_id, '1', t0, rv0[0:3], v2_i, v2
]

# conic flyby
sol_line_2_raw = [
    '0', '0', t0, rv0[0:3], v2_i, [0.0, 0.0, 0.0]
]

sol_line_3_raw = [
    '0', '0', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, [0.0, 0.0, 0.0]
]

# encounter with rogue1
sol_line_4_raw = [
    rogue1_id, '1', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, rel_v
]

In [22]:
def flatten_list(l):
    out = []
    for v in l:
        if isinstance(v, (int, float, str)):
            out.append(float(v))
        elif isinstance(v, (list, np.ndarray)):
            for y in v:
                out.append(float(y))
    return out

In [23]:
sol_lines = list()
sol_lines.append(flatten_list(sol_line_1_raw))
sol_lines.append(flatten_list(sol_line_2_raw))
sol_lines.append(flatten_list(sol_line_3_raw))
sol_lines.append(flatten_list(sol_line_4_raw))

In [24]:
sol_lines

[[10.0,
  1.0,
  2209032000.0,
  -20731752896.68304,
  1596950414.5640779,
  -4192053304.1797414,
  3.857374958498402,
  -0.35566128955943954,
  1.4471417684715098,
  3.9485112737168584,
  1.2726238199136557,
  2.768303784880207],
 [0.0,
  0.0,
  2209032000.0,
  -20731752896.68304,
  1596950414.5640779,
  -4192053304.1797414,
  3.857374958498402,
  -0.35566128955943954,
  1.4471417684715098,
  0.0,
  0.0,
  0.0],
 [0.0,
  0.0,
  4794018749.308968,
  -9116179240.167223,
  558404344.7941144,
  -204139662.95174012,
  5.6161222395839845,
  -0.47712011533007914,
  1.6430060244193552,
  0.0,
  0.0,
  0.0],
 [9.0,
  1.0,
  4794018749.308968,
  -9116179240.167223,
  558404344.7941144,
  -204139662.95174012,
  5.6161222395839845,
  -0.47712011533007914,
  1.6430060244193552,
  5.545872529788472,
  -4.541560221212665,
  1.3034806253301987]]

In [25]:
import shutil
import csv

shutil.copyfile('planetx_intercept.csv', 'rogue1_intercept.csv')

with open('rogue1_intercept.csv', 'a', newline='') as f:
    # header = ['#body_id','flag','epoch','rx','ry','rz','vx','vy','vz','ux','uy','uz']
    writer = csv.writer(f)
    # writer.writerow(header)
    for l in sol_lines:
        writer.writerow(l)